# FeedForward Neural Network

## Imports

In [7]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import re
import tensorflow as tf
from collections import defaultdict, Counter
from sklearn.model_selection import train_test_split
from typing import List, Tuple, Dict
import nltk
import os
import optuna
import sys
import pickle
import random
import ast
import math

sys.path.append(os.path.abspath(".."))

# Preprocessing
from base_preprocessing import BasePreprocessing
from bpe_generator_preprocessing import BPEGeneratorPreprocessing
from autotokenizer_generator_preprocessing import AutotokenizerGeneratorPreprocessing
from keras_tokenizer_generator_preprocessing import KerasTokenizerGeneratorPreprocessing
from base_generator_preprocessing import BaseGeneratorPreprocessing
# Models
from feedforward_nn import BatchFeedForwardNN
from optimized_feedforward_nn import OptimizedBatchFeedForwardNN

## ⭐ Base Preprocessing

In [8]:
if not os.path.exists("../../data/preprocessed_recipe.csv"):
    base_preprocessor = BasePreprocessing()
    raw_data = base_preprocessor.load('../../data/RAW_recipes.csv')
    data = base_preprocessor.transform(raw_data)
    data.to_csv("../../data/preprocessed_recipe.csv")
else:
    data = pd.read_csv("../../data/preprocessed_recipe.csv")
data.head()

,Unnamed: 0,id,name,minutes,n_steps,description,n_ingredients,steps_string_standardize,ingredients_text,tags_text,cuisine,calories,total_fat,sugar,sodium,protein,saturated_fat,carbohydrates
0,0,137739,arriba baked squash mexican,55,11,autumn is my favorite time of year to cook! th...,7,make a choic and proceed with recip depend on ...,"['winter squash', 'mexican seasoning', 'mixed ...","['60-minutes-or-less', 'time-to-make', 'course...",North America – United States,51.5,0.0,13.0,0.0,2.0,0.0,4.0
1,1,31490,breakfast pizza,30,9,this recipe calls for the crust to be prebaked...,6,preheat oven to 103.33 celsius °c press dough ...,"['prepared pizza crust', 'sausage patty', 'egg...","['30-minutes-or-less', 'time-to-make', 'course...",North America – United States,173.4,18.0,0.0,17.0,22.0,35.0,1.0
2,4,44061,amish tomato ketchup canning,190,5,my dh's amish mother raised him on this recipe...,8,"mix all ingredients& boil for 2 30.0 minute , ...","['tomato juice', 'apple cider vinegar', 'sugar...","['weeknight', 'time-to-make', 'course', 'main-...",North America – United States,352.9,1.0,337.0,23.0,3.0,0.0,28.0
3,5,25274,marinated olive,15,4,my italian mil was thoroughly impressed by my ...,9,toast the fennel seed and lightli crush them p...,"['fennel seeds', 'green olives', 'ripe olives'...","['15-minutes-or-less', 'time-to-make', 'course...",North America – United States,380.7,53.0,7.0,24.0,6.0,24.0,6.0
4,6,67888,barbecued rib,120,10,this recipe is posted by request and was origi...,22,in a medium saucepan combin all the ingredi fo...,"['pork spareribs', 'soy sauce', 'fresh garlic'...","['weeknight', 'time-to-make', 'course', 'main-...",North America – United States,1109.5,83.0,378.0,275.0,96.0,86.0,36.0


## Models

#### Training Process (context of 3)

Input: `<s> ing1 ing2 <STEPS> w1 w2 </s>`

| Sample | Input (`x`)                 | Target token (`y`) |
|------|-----------------------------|--------------|
| x1   | `<s> ing1 ing2`             | `<STEPS>`    |
| x2   | `ing1 ing2 <STEPS>`         | `w1`         |
| x3   | `ing2 <STEPS> w1`           | `w2`         |
| x4   | `<STEPS> w1 w2`             | `</s>`       |

Unknown words are encoded as `<UNKNOWN>`

#### Model Architecture

- One Hot encoding (`vocab_size`)
- Embdedding layer (`vocab_size`, `embedding_dim`)
- Dense layer (`embedding_dim`, `hidden_dim`)
- Dropout (if enabled)
- Dense layer (`hidden_dim`, `vocab_size`)
- Softmax layer (`vocab_size`)

### Simple FFNN

This is a simple feed forward neural network that learns to predict the next token using a fixed context

#### Preprocessing

In [3]:
base_generator_preprocessing = BaseGeneratorPreprocessing()
ingredients, steps = base_generator_preprocessing.fit_transform(data)

#### Training

In [7]:
model = BatchFeedForwardNN()
model.fit(ingredients[:10000], steps[:10000], embedding_dim=32, hidden_dim=64, context=30)
model.save("ffnn_save/ffnn_32_embedding_64_hidden_30_context.keras")

Vocab size is: 59349


/home/epita/nlp1/.venv-tf/lib/python3.11/site-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ w2v_embedding (Embedding)       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ hidden (Dense)                  │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/10


2025-05-08 16:18:43.610576: W tensorflow/core/framework/op_kernel.cc:1857] OP_REQUIRES failed at sparse_xent_op.cc:103 : INVALID_ARGUMENT: Received a label value of 50084 which is outside the valid range of [0, 64).  Label values: 36204 33141 5492 31386 9663 38963 21281 25873 35562 17609 49848 7981 28282 50084 275 19072 44892 390 2 29408 29128 16765 7787 10949 460 36851 59 15013 28086 21577 24181 130
2025-05-08 16:18:43.610596: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: INVALID_ARGUMENT: Received a label value of 50084 which is outside the valid range of [0, 64).  Label values: 36204 33141 5492 31386 9663 38963 21281 25873 35562 17609 49848 7981 28282 50084 275 19072 44892 390 2 29408 29128 16765 7787 10949 460 36851 59 15013 28086 21577 24181 130
	 [[{{function_node __inference_one_step_on_data_1991}}{{node compile_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/SparseSoftmaxCrossEntropyWithLogits}}]]


InvalidArgumentError: Graph execution error:

Detected at node compile_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/SparseSoftmaxCrossEntropyWithLogits defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "/nix/store/8akpjsgw444pybsflsmzfkrmkcp2q53a-python3-3.11.8-env/lib/python3.11/site-packages/ipykernel_launcher.py", line 17, in <module>

  File "/nix/store/8akpjsgw444pybsflsmzfkrmkcp2q53a-python3-3.11.8-env/lib/python3.11/site-packages/traitlets/config/application.py", line 1046, in launch_instance

  File "/nix/store/8akpjsgw444pybsflsmzfkrmkcp2q53a-python3-3.11.8-env/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 739, in start

  File "/nix/store/8akpjsgw444pybsflsmzfkrmkcp2q53a-python3-3.11.8-env/lib/python3.11/site-packages/tornado/platform/asyncio.py", line 195, in start

  File "/nix/store/mh6gqjf1w1i6ln1a61psnz0gzdmwrspl-python3-3.11.8/lib/

#### Evaluate

In [10]:
model = BatchFeedForwardNN()
model.load("ffnn_save/ffnn_32_embedding_64_hidden_30_context.keras")

In [ ]:
pred_steps = model.predict_batch(ingredients[10000:10100])

In [7]:
pred_steps

[['salt and cumin cook for 5.0 minute pure soup add the rest of the bean to the soup combin the cornstarch with 1 0.5 tablespoon of water add the lemon and the cornstarch to the soup cook until thicken',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt'],
 ['garl',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt'],
 ['salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt'],
 ['lemon juic',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt

### FFNN with AutoTokenizer

The AutoTokenizer from hugging face as tokenizer will help reducing the vocabulary size

#### Autotokenizer Preprocessing

In [3]:
autotokenizer_generator_preprocessing = AutotokenizerGeneratorPreprocessing()
ingredients, steps = autotokenizer_generator_preprocessing.fit_transform(data)

In [4]:
print(steps[0])
print(ingredients[0])

['make a choic and proceed with recip depend on size of squash', 'cut into half or fourth remov seed for spici squash', 'drizzl oliv oil or melt butter over each cut squash piec season with mexican season mix ii for sweet squash', 'drizzl melt honey', 'but', 'grate piloncillo over each cut squash piec season with sweet mexican spice mix bake at 176.67 celsius', 'again depend on size', 'for 40.0 minute up to an ho', 'until a fork can easili pierc the skin be care not to burn the squash especi if you opt to use sugar or butter if you feel more comfort', 'cover the squash with aluminum foil the first half ho', 'give or tak', 'of bake if desir', 'season with salt']
['winter squash', 'mexican seasoning', 'mixed spice', 'honey', 'but', 'olive oil', 'salt']


#### Training

In [ ]:
model = BatchFeedForwardNN()
model.fit(ingredients[:10000], steps[:10000], embedding_dim=256, hidden_dim=128, context=10, dropout=0.2)
model.save("ffnn_save/autotokenizer_ffnn_256_embedding_128_hidden_10_context.keras")

Vocab size is: 59349


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ w2v_embedding (Embedding)       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ hidden (Dense)                  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/10
6225/6225 ━━━━━━━━━━━━━━━━━━━━ 956s 153ms/step - loss: 10.2542 - sparse_categorical_accuracy: 0.0120 - val_loss: 12.3664 - val_sparse_categorical_accuracy: 0.0131
Epoch 2/10
6225/6225 ━━━━━━━━━━━━━━━━━━━━ 980s 157ms/step - loss: 9.2082 - sparse_categorical_accuracy: 0.0137 - val_loss: 13.9578 - val_sparse_categorical_accuracy: 0.0135
Epoch 3/10
1049/6225 ━━━━━━━━━━━━━━━━━━━━ 14:21 166ms/step - loss: 8.2970 - sparse_categorical_accuracy: 0.0142

#### Evaluate

In [13]:
model = BatchFeedForwardNN()
model.load("ffnn_save/autotokenizer_ffnn_32_embedding_64_hidden_30_context.keras")

TypeError: <class 'keras.src.models.sequential.Sequential'> could not be deserialized properly. Please ensure that components that are Python object instances (layers, models, etc.) returned by `get_config()` are explicitly deserialized in the model's `from_config()` method.

config={'module': 'keras', 'class_name': 'Sequential', 'config': {'name': 'sequential', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None, 'shared_object_id': 139964748708112}, 'layers': [{'module': 'keras.layers', 'class_name': 'InputLayer', 'config': {'batch_shape': [None, 30], 'dtype': 'float32', 'sparse': False, 'ragged': False, 'name': 'input_layer'}, 'registered_name': None}, {'module': 'keras.layers', 'class_name': 'Embedding', 'config': {'name': 'w2v_embedding', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None}, 'input_dim': 59349, 'output_dim'

In [9]:
pred_steps = model.predict_batch(ingredients[10000:10100])

ValueError: Model not fitted!

In [7]:
pred_steps

[['salt and cumin cook for 5.0 minute pure soup add the rest of the bean to the soup combin the cornstarch with 1 0.5 tablespoon of water add the lemon and the cornstarch to the soup cook until thicken',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt'],
 ['garl',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt'],
 ['salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt'],
 ['lemon juic',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt

### FFNN with BPE Tokenizer

The BPE tokenizer reduce the number of words but chunk the word into smaller pieces

#### BPE Preprocessing

In [4]:
sample = 1000
bpe_generator_preprocessing = BPEGeneratorPreprocessing(vocab_size=512)
if os.path.exists(f"fffn_save/{bpe_generator_preprocessing.get_savepath('ffnn_save')}"):
    bpe_generator_preprocessing.load(f"fffn_save/{bpe_generator_preprocessing.get_savepath()}")
else:
    bpe_generator_preprocessing.fit(data.iloc[:sample])
ingredients, steps = bpe_generator_preprocessing.transform(data.iloc[:sample])
bpe_generator_preprocessing.save_in_dir("ffnn_save")


FITTING BPE TOKENIZER ON 1000 SAMPLES

  - Processed 100 rows, corpus size: 1534
  - Processed 200 rows, corpus size: 3248
  - Processed 300 rows, corpus size: 4758
  - Processed 400 rows, corpus size: 6269
  - Processed 500 rows, corpus size: 7839
  - Processed 600 rows, corpus size: 9543
  - Processed 700 rows, corpus size: 11095
  - Processed 800 rows, corpus size: 12896
  - Processed 900 rows, corpus size: 14504
Final corpus size: 16079 tokens
Sample corpus entries: ['winter squash', 'mexican seasoning', 'mixed spice', 'honey', 'but']

Training BPE tokenizer...
Building initial word frequencies...
Unique words in corpus: 7492
Top 5 most common words: [('salt', 442), ('on', 290), ('sug', 228), ('but', 209), ('eg', 197)]

Building initial alphabet...
Alphabet size: 44
Initial vocabulary size: 47

Initializing splits (words into characters)...

Starting BPE merge operations...

Iteration 1: Current vocab size = 47
  Computing pair frequencies...
  Best pair: ('e', ' ') with frequency

In [9]:
print(steps[0])
print(ingredients[0])

['ma', 'k', 'e ', 'a', ' ch', 'o', 'ic', ' and', ' p', 'roc', 'eed', ' w', 'it', 'h', ' r', 'ec', 'ip', ' d', 'ep', 'en', 'd ', 'on', ' si', 'z', 'e ', 'of', ' s', 'qu', 'as', 'h', 'c', 'ut', ' i', 'n', 'to', ' h', 'alf', ' o', 'r', ' f', 'our', 'th', ' r', 'em', 'ov', ' s', 'eed', ' f', 'or ', 'sp', 'ic', 'i', ' s', 'qu', 'as', 'h', 'd', 'ri', 'z', 'z', 'l', ' o', 'li', 'v', ' o', 'il', ' o', 'r', ' m', 'el', 't ', 'b', 'ut', 't', 'er', ' o', 'v', 'er', ' e', 'ac', 'h', ' c', 'ut', ' s', 'qu', 'as', 'h', ' pi', 'ec', ' s', 'eas', 'on', ' w', 'it', 'h', ' m', 'ex', 'ic', 'an', ' s', 'eas', 'on', ' m', 'ix', ' i', 'i', ' f', 'or ', 's', 'w', 'ee', 't ', 's', 'qu', 'as', 'h', 'd', 'ri', 'z', 'z', 'l', ' m', 'el', 't ', 'h', 'on', 'ey', 'b', 'ut', 'g', 'ra', 't', 'e ', 'p', 'il', 'on', 'c', 'ill', 'o', ' o', 'v', 'er', ' e', 'ac', 'h', ' c', 'ut', ' s', 'qu', 'as', 'h', ' pi', 'ec', ' s', 'eas', 'on', ' w', 'it', 'h', ' s', 'w', 'ee', 't ', 'm', 'ex', 'ic', 'an', ' sp', 'ic', 'e ', 'm', '

#### Training

In [ ]:
model = BatchFeedForwardNN()
model.fit(ingredients[:1000], steps[:1000], embedding_dim=32, hidden_dim=64, context=3, dropout=0.2, learning_rate=1e-1)
model.save("ffnn_save/bpe_ffnn_32_embedding_64_hidden_3_context.keras")

Vocab size is: 250


/home/epita/nlp1/.venv-tf/lib/python3.11/site-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
2025-05-08 18:40:50.245832: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
2025-05-08 18:40:50.245851: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:178] verbose logging is disabled. Rerun with verbose logging (usually --v=1 or --vmodule=cuda_diagnostics=1) to get more diagnostic output from this module
2025-05-08 18:40:50.245854: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:183] retrieving CUDA diagnostic information for host: nixos
2025-05-08 18:40:50.245856: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:190] hostname: nixos
2025-05-08 18:40:50.245957: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:197] libcuda reported ver

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ w2v_embedding (Embedding)       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ hidden (Dense)                  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


2025-05-08 18:40:50.933948: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 369700656 exceeds 10% of free system memory.


Epoch 1/10
221212/481382 ━━━━━━━━━━━━━━━━━━━━ 5:46 1ms/step - loss: 5.0230 - sparse_categorical_accuracy: 0.0400

#### Evaluate

In [13]:
model = BatchFeedForwardNN()
model.load("ffnn_save/bpe_ffnn_32_embedding_64_hidden_3_context.keras")

TypeError: <class 'keras.src.models.sequential.Sequential'> could not be deserialized properly. Please ensure that components that are Python object instances (layers, models, etc.) returned by `get_config()` are explicitly deserialized in the model's `from_config()` method.

config={'module': 'keras', 'class_name': 'Sequential', 'config': {'name': 'sequential', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None, 'shared_object_id': 139964748708112}, 'layers': [{'module': 'keras.layers', 'class_name': 'InputLayer', 'config': {'batch_shape': [None, 30], 'dtype': 'float32', 'sparse': False, 'ragged': False, 'name': 'input_layer'}, 'registered_name': None}, {'module': 'keras.layers', 'class_name': 'Embedding', 'config': {'name': 'w2v_embedding', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None}, 'input_dim': 59349, 'output_dim'

In [9]:
pred_steps = model.predict_batch(ingredients[10000:10100])

ValueError: Model not fitted!

In [7]:
pred_steps

[['salt and cumin cook for 5.0 minute pure soup add the rest of the bean to the soup combin the cornstarch with 1 0.5 tablespoon of water add the lemon and the cornstarch to the soup cook until thicken',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt'],
 ['garl',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt'],
 ['salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt'],
 ['lemon juic',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt

### Optimized FFNN with BPE Tokenizer

The standard FFNN naturally generates a very large training set, as it includes all non-unique {context}-grams. This often leads to overfitting and inflates the resulting probability estimates

#### BPE Preprocessing

In [3]:
sample = 1000
bpe_generator_preprocessing = BPEGeneratorPreprocessing(vocab_size=512)
if os.path.exists(f"fffn_save/{bpe_generator_preprocessing.get_savepath('ffnn_save')}"):
    bpe_generator_preprocessing.load(f"fffn_save/{bpe_generator_preprocessing.get_savepath()}")
else:
    bpe_generator_preprocessing.fit(data.iloc[:sample])
ingredients, steps = bpe_generator_preprocessing.transform(data.iloc[:sample])
bpe_generator_preprocessing.save_in_dir("ffnn_save")


FITTING BPE TOKENIZER ON 1000 SAMPLES

  - Processed 100 rows, corpus size: 1534
  - Processed 200 rows, corpus size: 3248
  - Processed 300 rows, corpus size: 4758
  - Processed 400 rows, corpus size: 6269
  - Processed 500 rows, corpus size: 7839
  - Processed 600 rows, corpus size: 9543
  - Processed 700 rows, corpus size: 11095
  - Processed 800 rows, corpus size: 12896
  - Processed 900 rows, corpus size: 14504
Final corpus size: 16079 tokens
Sample corpus entries: ['winter squash', 'mexican seasoning', 'mixed spice', 'honey', 'but']

Training BPE tokenizer...
Building initial word frequencies...
Unique words in corpus: 7492
Top 5 most common words: [('salt', 442), ('on', 290), ('sug', 228), ('but', 209), ('eg', 197)]

Building initial alphabet...
Alphabet size: 44
Initial vocabulary size: 47

Initializing splits (words into characters)...

Starting BPE merge operations...

Iteration 1: Current vocab size = 47
  Computing pair frequencies...
  Best pair: ('e', ' ') with frequency

In [4]:
print(steps[0])
print(ingredients[0])

['ma', 'k', 'e ', 'a', ' ch', 'o', 'ic', ' and', ' p', 'roc', 'eed', ' w', 'it', 'h', ' r', 'ec', 'ip', ' d', 'ep', 'en', 'd ', 'on', ' si', 'z', 'e ', 'of', ' s', 'qu', 'as', 'h', 'c', 'ut', ' i', 'n', 'to', ' h', 'alf', ' o', 'r', ' f', 'our', 'th', ' r', 'em', 'ov', ' s', 'eed', ' f', 'or ', 'sp', 'ic', 'i', ' s', 'qu', 'as', 'h', 'd', 'ri', 'z', 'z', 'l', ' o', 'li', 'v', ' o', 'il', ' o', 'r', ' m', 'el', 't ', 'b', 'ut', 't', 'er', ' o', 'v', 'er', ' e', 'ac', 'h', ' c', 'ut', ' s', 'qu', 'as', 'h', ' pi', 'ec', ' s', 'eas', 'on', ' w', 'it', 'h', ' m', 'ex', 'ic', 'an', ' s', 'eas', 'on', ' m', 'ix', ' i', 'i', ' f', 'or ', 's', 'w', 'ee', 't ', 's', 'qu', 'as', 'h', 'd', 'ri', 'z', 'z', 'l', ' m', 'el', 't ', 'h', 'on', 'ey', 'b', 'ut', 'g', 'ra', 't', 'e ', 'p', 'il', 'on', 'c', 'ill', 'o', ' o', 'v', 'er', ' e', 'ac', 'h', ' c', 'ut', ' s', 'qu', 'as', 'h', ' pi', 'ec', ' s', 'eas', 'on', ' w', 'it', 'h', ' s', 'w', 'ee', 't ', 'm', 'ex', 'ic', 'an', ' sp', 'ic', 'e ', 'm', '

#### Training

In [ ]:
model = OptimizedBatchFeedForwardNN()
model.fit(ingredients[:1000], steps[:1000], embedding_dim=32, hidden_dim=64, context=3, dropout=0.2, learning_rate=1e-1, batch_size=512)
model.save("ffnn_save/bpe_optimized_ffnn_32_embedding_64_hidden_3_context.keras")

#### Evaluate

In [13]:
model = BatchFeedForwardNN()
model.load("ffnn_save/autotokenizer_ffnn_32_embedding_64_hidden_30_context.keras")

TypeError: <class 'keras.src.models.sequential.Sequential'> could not be deserialized properly. Please ensure that components that are Python object instances (layers, models, etc.) returned by `get_config()` are explicitly deserialized in the model's `from_config()` method.

config={'module': 'keras', 'class_name': 'Sequential', 'config': {'name': 'sequential', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None, 'shared_object_id': 139964748708112}, 'layers': [{'module': 'keras.layers', 'class_name': 'InputLayer', 'config': {'batch_shape': [None, 30], 'dtype': 'float32', 'sparse': False, 'ragged': False, 'name': 'input_layer'}, 'registered_name': None}, {'module': 'keras.layers', 'class_name': 'Embedding', 'config': {'name': 'w2v_embedding', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None}, 'input_dim': 59349, 'output_dim'

In [9]:
pred_steps = model.predict_batch(ingredients[10000:10100])

ValueError: Model not fitted!

In [ ]:
pred_steps

### Optimized FFNN with AutoTokenizer

#### Keras Tokenizer Preprocessing

In [9]:
keras_generator_preprocessing = KerasTokenizerGeneratorPreprocessing()
ingredients, steps = keras_generator_preprocessing.fit_transform(data)

In [10]:
print(steps[0])
print(ingredients[0])

['make', 'a', 'choic', 'and', 'proceed', 'with', 'recip', 'depend', 'on', 'size', 'of', 'squash', 'cut', 'into', 'half', 'or', 'fourth', 'remov', 'seed', 'for', 'spici', 'squash', 'drizzl', 'oliv', 'oil', 'or', 'melt', 'butter', 'over', 'each', 'cut', 'squash', 'piec', 'season', 'with', 'mexican', 'season', 'mix', 'ii', 'for', 'sweet', 'squash', 'drizzl', 'melt', 'honey', 'but', 'grate', 'piloncillo', 'over', 'each', 'cut', 'squash', 'piec', 'season', 'with', 'sweet', 'mexican', 'spice', 'mix', 'bake', 'at', '176.67', 'celsius', 'again', 'depend', 'on', 'size', 'for', '40.0', 'minute', 'up', 'to', 'an', 'ho', 'until', 'a', 'fork', 'can', 'easili', 'pierc', 'the', 'skin', 'be', 'care', 'not', 'to', 'burn', 'the', 'squash', 'especi', 'if', 'you', 'opt', 'to', 'use', 'sugar', 'or', 'butter', 'if', 'you', 'feel', 'more', 'comfort', 'cover', 'the', 'squash', 'with', 'aluminum', 'foil', 'the', 'first', 'half', 'ho', 'give', 'or', 'tak', 'of', 'bake', 'if', 'desir', 'season', 'with', 'salt']


#### Training

In [ ]:
model = OptimizedBatchFeedForwardNN()
model.fit(ingredients[:10000], steps[:10000], embedding_dim=32, hidden_dim=64, context=3, dropout=0.2, batch_size=256)
model.save("ffnn_save/autotokenizer_optimized_ffnn_32_embedding_64_hidden_3_context.keras")

Preparing unique context-grams for training...
Found 68589 unique context-grams
Vocab size: 15095
Unique training samples: 5204623
Data preparation took 31.63 seconds
One sample: [8624 6199 3178] - 89
Building model...


/home/epita/nlp1/.venv-tf/lib/python3.11/site-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pooling                         │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ hidden (Dense)                  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/10
 7020/18298 ━━━━━━━━━━━━━━━━━━━━ 7:10 38ms/step - loss: 0.0695 - sparse_categorical_accuracy: 0.0106

#### Evaluate

In [13]:
model = BatchFeedForwardNN()
model.load("ffnn_save/autotokenizer_optimized_ffnn_32_embedding_64_hidden_3_context.keras")

TypeError: <class 'keras.src.models.sequential.Sequential'> could not be deserialized properly. Please ensure that components that are Python object instances (layers, models, etc.) returned by `get_config()` are explicitly deserialized in the model's `from_config()` method.

config={'module': 'keras', 'class_name': 'Sequential', 'config': {'name': 'sequential', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None, 'shared_object_id': 139964748708112}, 'layers': [{'module': 'keras.layers', 'class_name': 'InputLayer', 'config': {'batch_shape': [None, 30], 'dtype': 'float32', 'sparse': False, 'ragged': False, 'name': 'input_layer'}, 'registered_name': None}, {'module': 'keras.layers', 'class_name': 'Embedding', 'config': {'name': 'w2v_embedding', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None}, 'input_dim': 59349, 'output_dim'

In [9]:
pred_steps = model.predict_batch(ingredients[10000:10100])

ValueError: Model not fitted!

In [7]:
pred_steps

[['salt and cumin cook for 5.0 minute pure soup add the rest of the bean to the soup combin the cornstarch with 1 0.5 tablespoon of water add the lemon and the cornstarch to the soup cook until thicken',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt'],
 ['garl',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt'],
 ['salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt'],
 ['lemon juic',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt',
  'salt